## PhDAI733 Group Project: Machine Learning Classification of Coronary Artery Disease

**Project Team:** Jude Adenuga, Pranathy Anumula, Rashidah Carr, Gino Varghese

**Deliverable:** Hyperparameter Tuning, Model Optimization, Final Model Evaluation, Real-World Impact, and Ethical Evaluation

**Dataset:** Heart Disease, UCI Machine Learning Repository (all four hospital sites pooled: Cleveland, Hungarian, Switzerland, and VA Long Beach, 920 patients)

**How this notebook is organized**

| Section |   
|---|
| Introduction: Where Part 1 Left Off and What Part 2 Addresses |   
| 1. Setup and Imports |   
| 2. Accessing the Data |   
| 3. Data Cleaning, Audit, and Exploration of All Sites |   
| 4. Preprocessing Pipeline |   
| 5. Train-Test Split |   
| 6. Baseline and Candidate Models |   
| 7. Hyperparameter Tuning |
| 8. Final Model Evaluation |    
| 9. Ethical Considerations |  
| 10. Real-World Application |  
| 11. Team Collaboration Process |   
| 12. Final Thoughts and Conclusion |   





## Introduction: Where Part 1 Left Off and What Part 2 Addresses

### Clinical problem, restated

Coronary artery disease is the largest component of cardiovascular disease, the leading cause of death worldwide. Its definitive test, coronary angiography, is invasive and expensive, so clinicians decide who needs it from a first-line workup including history, symptoms, a resting ECG, blood work, and an exercise stress test. Our project asks whether a model built only from that first-line workup can estimate each patient's probability of angiographically significant disease (more than 50 percent narrowing in a major vessel) well enough to help a cardiology clinic decide who should go to angiography first.

### What Part 1 established

Part 1 audited and cleaned the Cleveland file (303 patients), built a leakage-safe preprocessing pipeline over 11 first-line predictors, and trained a Random Forest. The initial forest reached 80.0 percent test accuracy and 75.0 percent sensitivity, missing 7 of 28 diseased patients. Tuning the forest's tree settings for recall did not help (8 missed patients).   

Part 1 closed with three potential next steps, and Part 2 is built around them:

1. **Add a baseline and a second model**, and report ROC-AUC so models can be compared independently of the decision threshold (Sections 6 and 7).
2. **Tune the decision threshold**, because the tree settings were not the lever that controls missed patients (Section 7.3).
3. **Test the model on the other three hospitals** in the same UCI dataset. Part 2 pools all four hospitals into one dataset and reports results for each hospital separately (Section 8.5).

### Stakeholders

- **Cardiologists and primary-care clinicians** (primary stakeholder): want a risk score that supports their referral decision and gives a clear reason behind each score.
- **Hospital quality and safety leadership**: need evidence that the model does not miss disease (false negatives are the costly error) and that it performs consistently across patient groups.
- **Patients**: bear the consequences of both a missed diagnosis and an unnecessary invasive procedure. They also deserve an explainable outcome.



## 1. Setup and Imports
All libraries are imported and warnings are suppressed so that the outputs stay readable.



In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn xgboost --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# --- Set up logging (replaces print statements for traceability) ---
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)
logger = logging.getLogger(__name__)

# --- Core libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- scikit-learn modules ---
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_predict
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

# --- Plot settings ---
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

RANDOM_STATE = 42

logger.info('Setup complete.')

## 2. Accessing the Data
Part 1 pulled only the Cleveland file (303 patients) through `fetch_ucirepo(id=45)`. For Part 2 we go back to the UCI archive and download the processed data files from each of the four hospitals. Every file carries the same 14 columns, with `?` standing in for missing values.

| Site | Investigators | Expected Patients |
|---|---|---|
| Cleveland | Robert Detrano (Cleveland Clinic Foundation) | 303 |
| Hungarian | Andras Janosi (Hungarian Institute of Cardiology) | 294 |
| Switzerland | William Steinbrunn, Matthias Pfisterer (University Hospitals, Zurich and Basel) | 123 |
| VA Long Beach | Robert Detrano (V.A. Medical Center, Long Beach) | 200 |   


Once the four files are loaded, we apply the same cleaning steps from Part 1: binarize the `disease` target, drop `ca` and `thal` from the predictor set, audit missing-value rates per site for the remaining 11 features, and impute the gaps so that every hospital's patients can contribute to the model.

### The data
The UCI Heart Disease dataset was collected in 1988 from patients who were referred for coronary angiography. Four hospitals contributed data: Cleveland, Budapest, Zurich/Basel, and Long Beach VA. While the original records contain 76 attributes, most published studies utilize a processed subset of 14 attributes. The Cleveland file is the most commonly used because it is the most complete, encompassing data from 303 patients across 14 columns. It is important to note that the data comes from patients specifically undergoing angiography, which may not represent the general population. This distinction matters when evaluating whether the model can be effectively applied in other contexts.

### The columns worth knowing
| Column | Type | What it means |
|---|---|---|
| age | Number | Years |
| sex | Binary | 1 = male, 0 = female |
| cp | Category | Chest pain: 1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic |
| trestbps | Number | Resting blood pressure (mm Hg) |
| chol | Number | Serum cholesterol (mg/dl) |
| fbs | Binary | Fasting blood sugar above 120 mg/dl |
| restecg | Category | Resting ECG: 0 normal, 1 ST-T abnormality, 2 left ventricular hypertrophy |
| thalach | Number | Maximum heart rate reached on the exercise test |
| exang | Binary | Chest pain brought on by exercise |
| oldpeak | Number | ST depression on exercise relative to rest |
| slope | Category | Slope of the peak exercise ST segment: 1 up, 2 flat, 3 down |
| ca | Count | Major vessels (0 to 3) coloured by fluoroscopy |
| thal | Category | Thallium scan: 3 normal, 6 fixed defect, 7 reversible defect |
| num | Ranked | Angiographic diagnosis, 0 (none) to 4, what we predict |

In [ ]:
# Column names for the UCI Heart Disease processed files
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
    'ca', 'thal', 'num'
]

# UCI archive base URL
base_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/'

# Map each hospital to its filename
sites = {
    'Cleveland':     'processed.cleveland.data',
    'Hungarian':     'processed.hungarian.data',
    'Switzerland':   'processed.switzerland.data',
    'VA_Long_Beach': 'processed.va.data'
}

# Read each site's CSV and tag it with the hospital name
frames = []
for site_name, filename in sites.items():
    df = pd.read_csv(base_url + filename, names=column_names, na_values='?')
    df['site'] = site_name
    frames.append(df)
    logger.info(f'{site_name}: {len(df)} patients loaded')

raw_df = pd.concat(frames, ignore_index=True)
logger.info(f'Total raw patients: {len(raw_df)}')

## 3. Data Cleaning, Audit, and Exploration of All Sites

The target is binarized exactly as in Part 1 (`disease` = 1 when any major vessel shows more than 50 percent narrowing), the same 11 first-line predictors are kept, and each hospital's file is audited for missing values before the gaps are filled.

In [ ]:
# Create binary target: 1 = disease present, 0 = no disease
raw_df['disease'] = (raw_df['num'] > 0).astype(int)

# Define the 11 features we will use (ca and thal excluded, same as Part 1)
continuous_features  = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['cp', 'restecg', 'slope']
binary_features      = ['sex', 'fbs', 'exang']
feature_columns      = continuous_features + categorical_features + binary_features

# Audit missing values in each site
logger.info('Missing values per site (11 model features):')
logger.info('=' * 70)
for site_name in sites:
    sub = raw_df.loc[raw_df['site'] == site_name, feature_columns]
    missing = sub.isna().sum()
    gaps = missing[missing > 0]
    if gaps.empty:
        logger.info(f'  {site_name} (n={len(sub)}): no missing values')
    else:
        logger.info(f'  {site_name} (n={len(sub)}):')
        for col, cnt in gaps.items():
            logger.info(f'    {col}: {cnt} missing ({100*cnt/len(sub):.1f}%)')

total_missing = raw_df[feature_columns].isna().sum()
logger.info(f'Total missing across all sites:\n{total_missing[total_missing > 0].to_string()}')

Cleveland has no gaps in any of the 11 model features. The other three hospitals are a different story. Hungarian data is mostly complete, but "slope" is absent in nearly 65 percent of its 294 records; cholesterol is missing in about 8 percent, and fasting blood sugar in roughly 3 percent. Switzerland's biggest gap is fbs, missing from 61 percent of records, with slope absent in about 14 percent. VA Long Beach is the hardest-hit site: trestbps, thalach, oldpeak, and exang are each missing in 26-28 percent of rows, and slope is gone from over half the records.

Across all 920 patients, slope leads the missing-value count at 309 entries (~34 percent), followed by fbs (90), oldpeak (62), trestbps (59), thalach and exang (55 each), cholesterol (30), and restecg (2).

Dropping every row with a gap would gut the non-Cleveland hospitals and undo the whole point of expanding the dataset. Instead, we impute: medians for continuous features and modes for categorical or binary ones. We do this within each site so that Cleveland's median cholesterol does not leak into the Hungarian records.

In [ ]:
# Placeholder zeros: no living patient has a cholesterol or resting blood pressure of 0.
# These values do not show up in the missing-value count above, so they are counted here.
zero_counts = raw_df.groupby('site')[['chol', 'trestbps']].agg(lambda s: int((s == 0).sum()))
logger.info(f'Values recorded as 0 (stand-ins for missing measurements):\n{zero_counts.to_string()}')

**Placeholder zeros.** The count above only finds values marked `?`. Two hospitals also used 0 as a stand-in for a missing cholesterol reading: all 123 Swiss patients and 49 of the 200 VA patients have a recorded cholesterol of 0, and one VA patient has a resting blood pressure of 0. The model keeps these values as recorded, so they are treated as real measurements. Because cholesterol is 0 only at these two high-prevalence hospitals, this is a data-quality limitation that Sections 8.6 and 9 return to.

In [ ]:
# Within-site imputation:
#   Continuous  -> median (robust to outliers)
#   Categorical -> mode  (most frequent value)

model_df = raw_df.copy()

for site_name in sites:
    mask = (model_df['site'] == site_name)

    for col in continuous_features:
        if model_df.loc[mask, col].isna().any():
            fill = model_df.loc[mask, col].median()
            model_df.loc[mask, col] = model_df.loc[mask, col].fillna(fill)

    for col in categorical_features + binary_features:
        if model_df.loc[mask, col].isna().any():
            fill = model_df.loc[mask, col].mode().iloc[0]
            model_df.loc[mask, col] = model_df.loc[mask, col].fillna(fill)

# Quick sanity checks
remaining = model_df[feature_columns].isna().sum().sum()
logger.info(f'Missing values after imputation: {remaining}')

before = len(model_df)
model_df = model_df.dropna(subset=['disease']).copy()
logger.info(f'Rows dropped (missing target): {before - len(model_df)}')

# Dataset summary
logger.info(f'Final dataset: {len(model_df)} patients')
logger.info(f'Patients per site:\n{model_df["site"].value_counts().to_string()}')
logger.info(f'Disease prevalence per site:\n{model_df.groupby("site")["disease"].agg(["size","mean"]).round(3).to_string()}')
logger.info(f'Overall disease prevalence: {model_df["disease"].mean():.2%}')

With the within-site imputation finished, no gaps remain in the 11 model features and no rows were lost to a missing target. The final combined set is 920 patients: Cleveland 303, Hungarian 294, VA Long Beach 200, Switzerland 123.

Disease prevalence varies sharply across sites. Cleveland (45.9 percent) and Hungarian (36.1 percent) sit closer to what you would expect from a broad referral base that mixes symptomatic patients with routine screens. Switzerland (93.5 percent) and VA Long Beach (74.5 percent) are dominated by patients who were already under heavy clinical suspicion when they arrived. The pooled prevalence of 55.3 percent gives us a near-balanced binary split, so we can train and evaluate without class-weight adjustments or resampling.



### 3.1 Exploring the Pooled Data

With the four hospitals combined, the data are explored in the same three ways as Part 1: the spread of a single variable (Figure 3a), a comparison between outcome groups (Figure 3b), and disease rates across categories (Figure 3c).

In [ ]:
# Figure 3a. Distribution of age (histogram)
# Figure 3b. Max heart rate by outcome (box and whiskers)


plot_df = model_df.assign(outcome=np.where(model_df['disease'] == 1, 'Disease', 'No disease'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=plot_df, x='age', bins=20, ax=axes[0])
axes[0].set_title('Figure 3a. Distribution of age (histogram)')
sns.boxplot(data=plot_df, x='outcome', y='thalach', order=['No disease', 'Disease'], ax=axes[1])
axes[1].set_title('Figure 3b. Max heart rate by outcome (box and whiskers)')
axes[1].set_xlabel('')
axes[1].set_ylabel('thalach')
plt.tight_layout()
plt.show()

print('Quartiles of max heart rate by outcome:')
print(plot_df.groupby('outcome')['thalach'].describe()[['25%', '50%', '75%']].round(1))


In [ ]:
# Figure 3c. Disease rate by hospital, chest pain type, sex, and exercise-induced angina
overall = 100 * model_df['disease'].mean()
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
panels = [('site', 'hospital'), ('cp', 'chest pain type'), ('sex', 'sex (1 = male)'), ('exang', 'exercise-induced angina')]
for ax, (col, label) in zip(axes, panels):
    rate = 100 * model_df.groupby(col)['disease'].mean()
    counts = model_df.groupby(col).size()
    if col != 'site':
        rate.index = rate.index.astype(int)
    rate.plot(kind='bar', ax=ax, rot=0 if col != 'site' else 20, edgecolor='black')
    ax.axhline(overall, color='red', ls='--', lw=1)
    ax.set_title(f'Disease rate by {label}')
    ax.set_ylabel('Disease rate (%)')
    ax.set_xlabel('')
    ax.set_ylim(0, 105)
    for i, v in enumerate(rate.values):
        ax.text(i, v + 2, 'n=%d' % counts.iloc[i], ha='center', fontsize=8)
fig.suptitle('Figure 3c. Disease rate across categories (dashed line = overall rate)')
plt.tight_layout()
plt.show()

for col in ['cp', 'sex', 'exang']:
    print(model_df.groupby(col)['disease'].agg(['size', 'mean']).round(3), '\n')
print('Women by hospital:', model_df[model_df['sex'] == 0]['site'].value_counts().to_dict())

**What the charts show.** Patients with disease are older and reach a lower peak heart rate than patients without it (Figure 3b): the median maximum heart rate is 125 beats per minute for diseased patients against 150 for disease-free patients, and the middle halves of the two groups barely overlap. The category comparisons repeat the Part 1 patterns at a larger scale. Asymptomatic chest pain carries the highest disease rate (79 percent), exercise-induced angina raises the rate from 36 to 81 percent, and men have a disease rate of 63 percent against 26 percent for women.

Two features of the pooled data shape the rest of the analysis. The first is that disease rates differ sharply by hospital, from 36 percent in Hungary to 94 percent in Switzerland, which reflects how sick patients were when they were referred more than any difference in biology. The second is that the sample is 79 percent male (726 men, 194 women), and 178 of the 194 women come from Cleveland and Hungary. The model therefore sees very few women from the high-prevalence hospitals, a point Section 9 returns to.

## 4. Preprocessing Pipeline

The preprocessing pipeline keeps the same structure as Part 1: continuous measurements are standardized, categorical codes are one-hot encoded with the first level dropped, and binary flags pass through unchanged. The hospital label is kept aside so that results can be broken down by site in Section 8.

In [ ]:
# Separate features and target
X = model_df[feature_columns].copy()
y = model_df['disease'].copy()
site_labels = model_df['site'].copy()   # kept for fairness analysis later

# Preprocessing pipeline (same structure as Part 1):
#   - Scale continuous features
#   - One-hot encode categorical features (drop first to avoid collinearity)
#   - Pass binary features through as-is
preprocessor = ColumnTransformer([
    ('cont', StandardScaler(), continuous_features),
    ('cat',  OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features),
    ('bin',  'passthrough', binary_features)
], remainder='drop')

logger.info(f'Predictor matrix: {X.shape}  |  Target: {y.name}')

## 5. Train-Test Split

In [ ]:
# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

# Keep site labels aligned for later per-site evaluation
site_train = site_labels.loc[X_train.index]
site_test  = site_labels.loc[X_test.index]

logger.info(f'Training: {len(X_train)} patients  |  Test: {len(X_test)} patients')
logger.info(f'Disease prevalence  train: {y_train.mean():.2%}  |  test: {y_test.mean():.2%}')
logger.info(f'Test set by site:\n{site_test.value_counts().to_string()}')

We split the 920 patients 80/20 using stratification on the target, which locks disease prevalence at essentially the same rate in both halves (55.30 percent training, 55.43 percent test). The test set of 184 patients includes representation from all four hospitals (Hungarian 64, Cleveland 55, VA Long Beach 37, Switzerland 28), so the evaluation ahead will reflect multi-site performance rather than the single-institution picture we had in Part 1.

## 6. Baseline and Candidate Models

Part 1 trained a default Random Forest on Cleveland data and ran a single grid search focused on disease recall. That tuned model actually hurt the held-out scores: accuracy dropped from 80.0 to 76.7 percent and false negatives went up from 7 to 8.

Here we take a wider approach in three ways:

1. **Broader model search.** Four classifiers go head to head (Logistic Regression, Random Forest, Gradient Boosting, and XGBoost), each with its own hyperparameter grid.
2. **ROC-AUC as the selection metric.** Rather than chasing recall in isolation, we rank models by ROC-AUC, which measures ranking quality across every possible threshold. Recall, precision, and F1 are still reported for the disease class.
3. **Threshold sweep.** Once the best estimator is picked, we sweep the probability threshold on cross-validated predictions to find the operating point that maximizes disease-class F1. Part 1 showed that the default 0.5 cutoff may not be the sweet spot.

A majority-class baseline and the default-settings score of each classifier (Section 6.2) give the reference points that tuning has to beat. All tuning runs on 5-fold stratified cross-validation within the training set. The test set stays sealed until Section 8.

In [ ]:
# --- 6.1 Four candidate classifiers, each with a hyperparameter grid ---

pipelines = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=RANDOM_STATE))
    ]),
    'GradientBoosting': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', GradientBoostingClassifier(random_state=RANDOM_STATE))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE))
    ])
}

param_grids = {
    'LogisticRegression': {
        'classifier__C':       [0.01, 0.1, 1, 10],
        'classifier__penalty': ['l2'],
        'classifier__solver':  ['lbfgs']
    },
    'RandomForest': {
        'classifier__n_estimators':      [100, 200, 300],
        'classifier__max_depth':         [3, 5, 10, None],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf':  [1, 2]
    },
    'GradientBoosting': {
        'classifier__n_estimators':  [100, 200],
        'classifier__max_depth':     [3, 5],
        'classifier__learning_rate': [0.05, 0.1, 0.2],
        'classifier__subsample':     [0.8, 1.0]
    },
    'XGBoost': {
        'classifier__n_estimators':     [100, 200],
        'classifier__max_depth':        [2, 3],
        'classifier__learning_rate':    [0.03, 0.05],
        'classifier__subsample':        [0.8, 1.0],
        'classifier__colsample_bytree': [0.8, 1.0]
    }
}

# 5-fold stratified CV used throughout
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

logger.info('Candidate models defined:')
for name, grid in param_grids.items():
    combos = 1
    for vals in grid.values():
        combos *= len(vals)
    logger.info(f'  {name}: {combos} combinations')

Four candidate classifiers are set up, each wrapped in a pipeline with the shared preprocessor. Logistic Regression has the most compact search space at 4 combinations, varying only the regularization strength (C). Random Forest has the largest grid at 48 combinations, covering the number of trees, maximum depth, and the split and leaf controls. Gradient Boosting has 24 combinations covering learning rate, depth, tree count, and row subsampling, and XGBoost adds 32 combinations covering its own learning rate, depth, subsampling, and column sampling.   

In total that is 108 configurations, each evaluated with 5-fold stratified cross-validation scored on ROC-AUC, for 540 fits before the final model is chosen.

In [ ]:
# --- 6.2 Majority-class baseline and default-settings scores ---
# The baseline predicts "disease" (the larger class) for every patient. Every real model must beat
# its ROC-AUC of 0.50. The default-settings scores show how much the tuning in Section 7 adds.
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

baseline = Pipeline([('preprocessor', preprocessor),
                     ('classifier', DummyClassifier(strategy='most_frequent'))])
baseline_auc = cross_val_score(baseline, X_train, y_train, cv=cv, scoring='roc_auc').mean()
baseline_acc = cross_val_score(baseline, X_train, y_train, cv=cv, scoring='accuracy').mean()
logger.info(f'Majority-class baseline: CV ROC-AUC {baseline_auc:.4f}  |  CV accuracy {baseline_acc:.2%}')

default_auc = {}
for name in pipelines:
    default_auc[name] = cross_val_score(pipelines[name], X_train, y_train, cv=cv, scoring='roc_auc').mean()
    logger.info(f'{name} (default settings): CV ROC-AUC {default_auc[name]:.4f}')

The majority-class baseline predicts disease for every patient. That gives about 55 percent accuracy, simply because 55 percent of patients have disease, but an ROC-AUC of exactly 0.50, because it cannot rank one patient above another. At their default settings all four classifiers clear that floor by a wide margin, with Logistic Regression at about 0.868, Random Forest at about 0.857, and Gradient Boosting at about 0.853, and XGBoost at about 0.832.   

These default scores are the reference for judging how much the tuning in Section 7 improves each model.

## 7. Hyperparameter Tuning

Each candidate is tuned with `GridSearchCV` on the training set (Section 7.1), the results are compared in one table (Section 7.2), and the decision threshold of the final model is then tuned on cross-validated predictions (Section 7.3).

In [ ]:
# --- 7.1 Grid search: find the best hyperparameters for each model ---
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

best_models = {}

for name in pipelines:
    logger.info(f'Tuning {name} ...')
    search = GridSearchCV(
        pipelines[name],
        param_grids[name],
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train, y_train)
    best_models[name] = search
    logger.info(f'  Best CV ROC-AUC: {search.best_score_:.4f}')
    logger.info(f'  Best params: {search.best_params_}')

# Highest cross-validated score among the four models
top_name = max(best_models, key=lambda k: best_models[k].best_score_)
logger.info(f'Highest CV ROC-AUC: {top_name} ({best_models[top_name].best_score_:.4f})')

# Pick the overall winner
winner_name = max(best_models, key=lambda k: best_models[k].best_score_)
winner_gs   = best_models[winner_name]
logger.info(f'Best overall model: {winner_name} (CV ROC-AUC = {winner_gs.best_score_:.4f})')

# Team decision: XGBoost scores slightly higher, but the gap is far smaller than the
# variation between cross-validation folds, so the tuned Random Forest is kept as the final model.
winner_name = 'RandomForest'
winner_gs   = best_models[winner_name]
logger.info(f'Final model selected by the team: {winner_name} (CV ROC-AUC = {winner_gs.best_score_:.4f})')

All four classifiers performed closely, with cross-validated ROC-AUC scores less than one percentage point apart. Logistic Regression reached 0.8700 with moderate regularization (C = 0.1, L2), which shows that a simple linear boundary already captures most of the signal in the 11 features. Random Forest scored 0.8716 with a shallow configuration (maximum depth 5, 100 trees), so a modest amount of non-linear modeling helps without overfitting. Gradient Boosting recorded 0.8653 under a conservative setup with a slow learning rate (0.05), shallow trees (depth 3), and 80 percent row subsampling. XGBoost scored highest at 0.8719, also with a conservative setup that used a learning rate of 0.03, a maximum depth of 2, and 80 percent column sampling for each tree.

Although XGBoost scored slightly higher, the team kept the tuned Random Forest as the final model. The two differ by only 0.0003, which is far smaller than the variation between cross-validation folds, so the difference is not meaningful. The Random Forest’s results are evaluated in full in Section 8, and its behavior is explained through its feature importances in Section 8.6.

One limitation applies to all four models. Missing values were filled before the data were split (Section 3), so test patients’ values contributed to the hospital medians used during training. This may make every score slightly optimistic, and future work should fill missing values inside the training pipeline.

In [ ]:
# --- 7.2 Summary table of cross-validation results ---

cv_summary = pd.DataFrame({
    'Model':           list(best_models.keys()),
    'Best CV ROC-AUC': [best_models[k].best_score_ for k in best_models],
    'Best Params':     [str(best_models[k].best_params_) for k in best_models]
}).sort_values('Best CV ROC-AUC', ascending=False).reset_index(drop=True)

logger.info('Cross-Validation Results (sorted by ROC-AUC):')
logger.info(f'\n{cv_summary.to_string(index=False)}')

The summary table lists all four tuned models. XGBoost (0.8719) and Random Forest (0.8716) share the top, followed by Logistic Regression (0.8700) and Gradient Boosting (0.8653). The sub-one-point gap across all four reinforces the point that, at this data scale and with these features, the choice of algorithm matters less than the quality of the inputs. The tuned Random Forest goes forward to the threshold sweep below.


In [ ]:
# --- 7.2b Did tuning improve the models? ---
gain = pd.DataFrame({
    'Model': list(default_auc.keys()),
    'Default CV ROC-AUC': list(default_auc.values()),
    'Tuned CV ROC-AUC': [best_models[k].best_score_ for k in default_auc],
})
gain['Improvement'] = gain['Tuned CV ROC-AUC'] - gain['Default CV ROC-AUC']
gain.loc[len(gain)] = ['Majority baseline', baseline_auc, np.nan, np.nan]
logger.info(f'Default vs. tuned cross-validated ROC-AUC:\n{gain.round(4).to_string(index=False)}')

Tuning improved every model, and it helped the tree-based models most. The Random Forest gained about 0.015 in ROC-AUC (from about 0.857 to 0.872) once its trees were capped at a depth of 5, because shallower trees cannot memorize noise in the training data. XGBoost gained the most, about 0.040 (from about 0.832 to 0.872), because its default settings grow much deeper trees than this small dataset can support. Gradient Boosting gained about 0.012, and Logistic Regression gained only about 0.002, because its default settings were already close to the best ones. All of the tuned models finish far above the baseline's 0.50.

In [ ]:
# --- 7.3 Threshold sweep: find the cutoff that maximizes disease F1 ---

best_pipeline = winner_gs.best_estimator_

# Get cross-validated predicted probabilities on the training set
cv_probs = cross_val_predict(
    best_pipeline, X_train, y_train, cv=cv, method='predict_proba'
)[:, 1]

# Try every threshold from 0.20 to 0.79 and record the metrics
thresholds     = np.arange(0.20, 0.80, 0.01)
f1_scores      = []
recall_scores  = []
precision_scores = []

for t in thresholds:
    preds = (cv_probs >= t).astype(int)
    f1_scores.append(f1_score(y_train, preds))
    recall_scores.append(recall_score(y_train, preds))
    precision_scores.append(precision_score(y_train, preds, zero_division=0))

# Optimal threshold = highest F1
best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

logger.info(f'Optimal threshold: {best_threshold:.2f}')
logger.info(f'  Recall:    {recall_scores[best_idx]:.2%}')
logger.info(f'  Precision: {precision_scores[best_idx]:.2%}')
logger.info(f'  F1:        {f1_scores[best_idx]:.2%}')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores,        label='F1 (disease)',        linewidth=2)
ax.plot(thresholds, recall_scores,    label='Recall (disease)',    linewidth=1.5, linestyle='--')
ax.plot(thresholds, precision_scores, label='Precision (disease)', linewidth=1.5, linestyle=':')
ax.axvline(best_threshold, color='red', linewidth=1, label=f'Best = {best_threshold:.2f}')
ax.set_xlabel('Probability Threshold')
ax.set_ylabel('Score')
ax.set_title(f'{winner_name}: Threshold Sweep (CV Predictions)')
ax.legend()
ax.set_xlim(0.20, 0.80)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### 7.4 Tuning Summary

The threshold sweep on the tuned Random Forest's cross-validated probabilities lands on 0.42 as the best operating point, eight hundredths below the default 0.50. At 0.42 the disease recall jumps to 92.14 percent (the model catches more than 9 out of every 10 disease patients), while precision holds at 76.69 percent, giving a disease F1 of 83.71 percent.

The plot shows the mechanics of this trade-off. When we slide the cutoff down from 0.50 toward 0.42, recall (dashed line) climbs steeply while precision (dotted line) dips only a little. The F1 curve (solid line) peaks right at 0.42 and begins to fall once the precision losses outweigh the recall gains.

From a clinical standpoint, a lower threshold is defensible for a screening tool: a false negative here means a patient with coronary artery disease walks out without further workup, whereas a false positive just triggers an additional non-invasive test. The cost asymmetry favors catching more true positives even at the price of a few extra follow-ups. We will apply both the 0.42 and the 0.50 thresholds to the held-out test set in Section 8 so we can confirm whether this improvement holds on unseen data.

## 8. Final Model Evaluation

Now we take the tuned Random Forest to the held-out test set. With four hospitals pooled, this test set is roughly three times the size of Part 1's Cleveland-only split (184 vs. 60 patients), which makes the recall and precision estimates more stable. The analyses follow in Sections 8.1 to 8.6:

1. **Default (0.5) vs. tuned threshold:** does lowering the cutoff actually reduce false negatives on unseen data?
2. **Side-by-side confusion matrices:** a concrete look at where predictions shift between the two thresholds.
3. **ROC curve and AUC:** a threshold-independent view of the model's ranking quality.
4. **Fairness by sex and age (8.4) and by hospital site (8.5):** Part 1 Section 8 raised the concern that a male-heavy training set could hurt sensitivity for women, and Section 10 called for testing on the other hospitals. Both questions are addressed here.
5. **Feature importance (8.6):** do the features driving the model's decisions line up with what cardiologists actually rely on?

In [ ]:
# --- 8.1 Predict on the test set at both thresholds ---

test_probs     = best_pipeline.predict_proba(X_test)[:, 1]
y_pred_default = (test_probs >= 0.50).astype(int)
y_pred_tuned   = (test_probs >= best_threshold).astype(int)

logger.info(f'{winner_name}: Test Set Results')
logger.info('=' * 60)

for label, preds in [('0.50 (default)', y_pred_default),
                     (f'{best_threshold:.2f} (tuned)', y_pred_tuned)]:
    logger.info(f'\n--- Threshold = {label} ---')
    logger.info(f'Accuracy: {accuracy_score(y_test, preds):.2%}')
    logger.info(f'ROC-AUC:  {roc_auc_score(y_test, test_probs):.4f}')
    report = classification_report(
        y_test, preds, target_names=['No Disease', 'Disease'],
        digits=3, zero_division=0
    )
    logger.info(f'\n{report}')

In [ ]:
# --- 8.2 Confusion matrices: default vs. tuned threshold ---

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in zip(axes,
                             [y_pred_default, y_pred_tuned],
                             ['Threshold = 0.50', f'Threshold = {best_threshold:.2f}']):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease']).plot(
        cmap='Blues', values_format='d', ax=ax
    )
    ax.set_title(f'{winner_name}: {title}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Numeric comparison table
tn_d, fp_d, fn_d, tp_d = confusion_matrix(y_test, y_pred_default).ravel()
tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred_tuned).ravel()

comp = pd.DataFrame({
    'Metric': ['True Neg', 'False Pos', 'False Neg', 'True Pos',
               'Accuracy', 'Precision', 'Recall', 'F1'],
    'Default (0.50)': [tn_d, fp_d, fn_d, tp_d,
                       accuracy_score(y_test, y_pred_default),
                       precision_score(y_test, y_pred_default, zero_division=0),
                       recall_score(y_test, y_pred_default),
                       f1_score(y_test, y_pred_default)],
    f'Tuned ({best_threshold:.2f})': [tn_t, fp_t, fn_t, tp_t,
                       accuracy_score(y_test, y_pred_tuned),
                       precision_score(y_test, y_pred_tuned, zero_division=0),
                       recall_score(y_test, y_pred_tuned),
                       f1_score(y_test, y_pred_tuned)]
})

logger.info(f'Comparison:\n{comp.to_string(index=False)}')

The side-by-side matrices put the numbers in front of you. At 0.50, 62 of 82 healthy patients are correctly classified and 92 of 102 disease patients are caught, with 20 false positives and 10 false negatives. Moving to 0.42 recovers 3 of those false negatives (down to 7, true positives up to 95), but at the expense of 9 extra false positives (up to 29), pulling true negatives down to 53.

In concrete terms: the tuned threshold saves roughly 3 additional patients per 184 from being sent home without further workup, at the cost of sending 9 more healthy patients to a confirmatory test they do not need. Most screening guidelines would accept that trade-off, since a missed catheterization referral carries far graver consequences than an extra stress test.

In [ ]:
# --- 8.3 ROC Curve ---

fpr, tpr, _ = roc_curve(y_test, test_probs)
auc_val     = roc_auc_score(y_test, test_probs)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, linewidth=2, label=f'{winner_name} (AUC = {auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve: Test Set (All Sites)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

logger.info(f'Test-set ROC-AUC: {auc_val:.4f}')

The ROC curve hugs the upper-left corner, hitting roughly 60 percent true-positive rate before the false-positive rate has barely moved off zero, and crossing 90 percent recall at about a 15 percent false-positive rate. The test-set AUC of 0.9200 means that if you pick one diseased patient and one healthy patient at random, the model assigns a higher risk score to the diseased patient 92 percent of the time.

That is a step up from the Cleveland-only forest in Part 1 (test ROC-AUC 0.88), achieved on a test set that mixes four hospitals with different referral patterns and prevalence rates. Because the same hospitals also supplied the training patients, the result shows that the model works across this mix of sites. It does not yet show how the model would perform at a hospital it has never seen.

In [ ]:
# --- 8.4 Fairness check: performance by sex ---

# Build a working dataframe for the test set
test_df = X_test.copy()
test_df['y_true'] = y_test.values
test_df['y_pred'] = y_pred_tuned
test_df['y_prob'] = test_probs
test_df['site']   = site_test.values

logger.info('Performance by Sex (tuned threshold)')
logger.info('=' * 60)

for sex_val, sex_label in [(1, 'Male'), (0, 'Female')]:
    sub = test_df[test_df['sex'] == sex_val]
    n_dis = int(sub['y_true'].sum())

    if len(sub) == 0 or sub['y_true'].nunique() < 2:
        logger.info(f'{sex_label}: too few patients (n={len(sub)}, disease={n_dis})')
        continue

    acc = accuracy_score(sub['y_true'], sub['y_pred'])
    rec = recall_score(sub['y_true'], sub['y_pred'], zero_division=0)
    prec = precision_score(sub['y_true'], sub['y_pred'], zero_division=0)
    f1  = f1_score(sub['y_true'], sub['y_pred'], zero_division=0)
    auc = roc_auc_score(sub['y_true'], sub['y_prob'])
    tn, fp, fn, tp = confusion_matrix(sub['y_true'], sub['y_pred']).ravel()

    logger.info(f'--- {sex_label} (n={len(sub)}, disease={n_dis}) ---')
    logger.info(f'  Accuracy: {acc:.2%}  Recall: {rec:.2%}  Precision: {prec:.2%}  F1: {f1:.2%}')
    logger.info(f'  ROC-AUC: {auc:.4f}')
    logger.info(f'  TN={tn}  FP={fp}  FN={fn}  TP={tp}')

Discrimination is comparable across sex: ROC-AUC is 0.9081 for males and 0.8935 for females, a gap small enough to be sampling noise. Disease recall is high for both groups (93.6 percent for males, 87.5 percent for females), with only 6 missed cases among men and 1 among women.

The precision gap is the more striking finding. Among females, disease precision drops to 58.3 percent versus 78.6 percent for males: 5 of the 12 women flagged as disease-positive were actually healthy. Given that about four-fifths of the patients are male (Section 3.1), the model may be over-predicting disease in the smaller female subgroup. That said, only 35 women (8 with disease) make it into the test set, so these metrics carry wide confidence intervals. In fact the model flags a smaller share of healthy women (19 percent) than of healthy men (44 percent), so the lower precision mainly reflects the lower disease rate among women. A larger female cohort would be needed to confirm whether the precision difference reflects genuine bias or just small-sample instability.

In [ ]:
# --- 8.4b Fairness check: performance by age band (promised in Part 1, Section 8.2) ---
test_df['age_band'] = pd.cut(test_df['age'], [0, 45, 60, 120], labels=['45 and under', '46 to 60', 'Over 60'])

age_rows = []
for band, sub in test_df.groupby('age_band', observed=True):
    tn, fp, fn, tp = confusion_matrix(sub['y_true'], sub['y_pred'], labels=[0, 1]).ravel()
    age_rows.append({'Age band': band, 'n': len(sub), 'Disease': int(sub['y_true'].sum()),
                     'Recall': tp / (tp + fn), 'Missed (FN)': fn,
                     'Specificity': tn / (tn + fp), 'False alarms (FP)': fp,
                     'ROC-AUC': roc_auc_score(sub['y_true'], sub['y_prob'])})
age_df = pd.DataFrame(age_rows)
logger.info(f'Performance by age band (tuned threshold):\n{age_df.round(3).to_string(index=False)}')

Part 1 predicted that younger patients with disease would be the ones most likely to be missed, because youth and a high peak heart rate both lower the model's score. The age breakdown supports that prediction. Recall is lowest for patients aged 45 and under (3 of 19 diseased patients missed, about 84 percent recall) and highest for patients over 60 (1 of 35 missed, about 97 percent).

The opposite error grows with age. Specificity falls from about 82 percent in the youngest group to about 70 percent in the middle group and to about 15 percent for patients over 60, where 11 of the 13 disease-free patients are flagged. In practice, younger diseased patients are the group most likely to be sent home, and older healthy patients are the group most likely to be sent for an unnecessary procedure. The age bands are small (41, 95, and 48 test patients), so these are warning signs to monitor, not precise estimates.

In [ ]:
# --- 8.5 Performance by hospital site ---

logger.info('Performance by Hospital Site (tuned threshold)')
logger.info('=' * 65)

site_rows = []
for site_name in ['Cleveland', 'Hungarian', 'Switzerland', 'VA_Long_Beach']:
    sub = test_df[test_df['site'] == site_name]
    if len(sub) == 0:
        continue

    n_dis = int(sub['y_true'].sum())
    row = {'Site': site_name, 'n': len(sub), 'Disease': n_dis}
    row['Accuracy']  = accuracy_score(sub['y_true'], sub['y_pred'])
    row['Recall']    = recall_score(sub['y_true'], sub['y_pred'], zero_division=0)
    row['Precision'] = precision_score(sub['y_true'], sub['y_pred'], zero_division=0)
    row['F1']        = f1_score(sub['y_true'], sub['y_pred'], zero_division=0)

    if sub['y_true'].nunique() >= 2:
        row['ROC-AUC'] = roc_auc_score(sub['y_true'], sub['y_prob'])
    else:
        row['ROC-AUC'] = np.nan

    site_rows.append(row)

    cm = confusion_matrix(sub['y_true'], sub['y_pred'])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        logger.info(f'--- {site_name} (n={len(sub)}, disease={n_dis}) ---')
        logger.info(f'  Acc: {row["Accuracy"]:.2%}  Rec: {row["Recall"]:.2%}  F1: {row["F1"]:.2%}  AUC: {row["ROC-AUC"]:.4f}')
        logger.info(f'  TN={tn}  FP={fp}  FN={fn}  TP={tp}')

site_df = pd.DataFrame(site_rows)
logger.info(f'Summary:\n{site_df.round(3).to_string(index=False)}')

The per-site breakdown answers Part 1's third next step within the pooled design. Because every hospital also contributed training patients, it shows how the model performs at each site, not how it would perform at a hospital it has never seen. Cleveland and Hungarian, the two sites with a meaningful balance of disease and non-disease patients, show strong discrimination at AUC 0.922 and 0.919 respectively, with recall above 86 percent and accuracy in the mid-to-upper 70s.

Switzerland hits 96.4 percent accuracy and 100 percent recall, but that is less impressive than it looks: 27 of 28 test patients from that site have disease, so the model barely has to distinguish anything. The AUC of 0.778 reflects the fact that there is essentially only one healthy patient to rank against.

VA Long Beach shows 96.3 percent recall but the lowest AUC (0.759) and only 73 percent accuracy, driven by 9 false positives among just 10 healthy patients.

The bottom line: the model generalizes well across the two hospitals that have real class balance. The high-prevalence sites confirm the model's sensitivity but cannot tell us much about its ability to rule out disease. That is a function of the referral-heavy populations at those centers, not a flaw in the model.

In [ ]:
# --- 8.6 Feature importance ---

# Get feature names after preprocessing
feat_names = list(continuous_features)
cat_enc = best_pipeline.named_steps['preprocessor'].named_transformers_['cat']
feat_names += list(cat_enc.get_feature_names_out(categorical_features))
feat_names += binary_features

# Extract importances from the Random Forest
clf = best_pipeline.named_steps['classifier']
importances = clf.feature_importances_

imp_df = pd.DataFrame({
    'Feature': feat_names,
    'Importance': importances
}).sort_values('Importance', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp_df['Feature'], imp_df['Importance'], edgecolor='black')
ax.set_xlabel('Feature Importance')
ax.set_title(f'{winner_name}: Feature Importances (All Sites)')
plt.tight_layout()
plt.show()

logger.info(f'Feature importances:\n{imp_df.sort_values("Importance", ascending=False).to_string(index=False)}')

The importance ranking makes clinical sense. Chest pain type 4 (asymptomatic presentation) leads at 19.5 percent. Patients who show up without typical angina but test positive are strong candidates for occult coronary disease. Exercise-induced angina (exang, 13.7 percent), ST depression during exercise (oldpeak, 12.9 percent), and peak heart rate (thalach, 11.0 percent) follow, and all three are standard exercise-stress-test variables that cardiologists weigh heavily in initial workups. Cholesterol and age come next at around 10 and 9.5 percent.

Further down, other chest pain subtypes (cp_2.0, 7.8 percent), sex (5.5 percent), and resting blood pressure (4.2 percent) contribute modestly. Resting ECG, slope categories, and fasting blood sugar each account for less than 1.5 percent, which is consistent with the clinical literature: once you have the exercise variables and chest pain presentation, those features add little incremental value.

Two cautions apply. The model was trained on all four hospitals pooled, so these importances describe the combined data, not each hospital separately. Cholesterol's fairly high rank (about 10 percent) should also be read with care, because Switzerland and the VA recorded missing cholesterol as 0 (Section 3), so part of its importance may come from identifying those two high-prevalence hospitals instead of from the measurement itself.

### 8.7 Summary of the Technical Work (Sections 2 to 8)

**Expanded dataset.** We went from the 303-patient Cleveland subset in Part 1 to all four hospitals in the UCI Heart Disease repository (Cleveland 303, Hungarian 294, Switzerland 123, VA Long Beach 200), for a combined 920 patients. Missing values in the non-Cleveland sites were imputed within each hospital using medians for continuous features and modes for categorical ones, so no hospital's clinical norms leaked into another's records.

**Model selection.** Four classifiers (Logistic Regression, Random Forest, Gradient Boosting, and XGBoost) were tuned through grid search with 5-fold stratified cross-validation, scored on ROC-AUC. All four landed within half a percentage point of each other, suggesting a performance ceiling around 0.87 AUC for these 11 first-line features. XGBoost scored highest at 0.8719, essentially tied with the Random Forest at 0.8716, and the team kept the Random Forest as the final model. Tuning raised every model's score above its default setting, by about 0.015 for the Random Forest, and every model beat the majority-class baseline (0.50) by a wide margin.

**Threshold tuning.** A sweep of the probability cutoff on cross-validated predictions identified 0.42 as the point that maximizes disease-class F1. Compared to the default 0.50, this threshold trades a small drop in precision for a meaningful jump in recall, catching about 3 more disease patients per 184 while sending 9 more healthy patients to follow-up testing. In a screening context where missed disease is costlier than an extra stress test, that is a trade-off worth making.

**Test-set performance.** The tuned Random Forest posted a test-set ROC-AUC of 0.9200 on the multi-site holdout, showing strong performance across the mix of hospitals in the test set. Disease recall at the 0.42 threshold reached 93.1 percent.

**Fairness.** Discrimination was comparable across sex (AUC 0.91 male, 0.89 female), though disease precision was lower for females, possibly a reflection of the mostly male training set, possibly just small-sample noise with only 35 women in the test split. By age, recall was lowest for patients aged 45 and under (84 percent) and specificity was lowest for patients over 60 (15 percent). Per-site evaluation showed strong results for the balanced-prevalence hospitals (Cleveland AUC 0.922, Hungarian 0.919) and confirmed the model's sensitivity at the high-prevalence sites, even if those sites cannot tell us much about specificity.

**Feature importance.** The top predictors (asymptomatic chest pain type, exercise-induced angina, ST depression, and peak heart rate) are the same variables that cardiologists lean on in practice. Cholesterol's rank should be read with care because of the placeholder zeros noted in Section 3.

**Limitations.** Within-site imputation fills gaps but introduces its own uncertainty, especially for features like "slope" that are missing in over half of some hospital's records. The four datasets date to the 1980s, and clinical protocols have changed since then. Referral bias affects all sites. The female subsample is small. Missing values were filled before the train-test split, which may make the reported scores slightly optimistic. Despite these caveats, the multi-site evaluation gives us the first quantitative evidence that a first-line-workup model can perform well across four hospitals with very different patient populations.

## 9. Ethical Considerations
- What potential biases exist in the dataset?
- How would the model perform across different demographic groups?
- Is the model interpretable and fair?

Each question is answered below with evidence from this notebook.

### 9.1 What potential biases exist in the dataset? Could the model reflect or reinforce them?

Three biases are visible in the data before any model is trained.

**Sex imbalance.** The pooled sample is 79 percent male (726 men and 194 women), and disease prevalence is 63 percent in men against 26 percent in women (Section 3.1). The model therefore learns from 459 diseased men but only 50 diseased women. The imbalance is also uneven across hospitals, since 178 of the 194 women come from Cleveland and Hungary and only 16 come from Switzerland and the VA. Clinical research has documented that coronary disease is under-recognized in women, in part because their symptoms often differ from the classic male presentation (Garcia et al., 2016). A model trained on this sample could carry that pattern forward.

**Referral bias.** Every patient in all four files was referred for angiography, so the sample is not the general population. It is the subset of patients whose clinicians already suspected disease. The clearest sign is that asymptomatic patients have the highest disease rate (79 percent), because an asymptomatic patient reaches angiography only when another test was abnormal. Disease rates also range from 36 percent in Hungary to 94 percent in Switzerland, which reflects how each hospital referred patients. The model learns these referral patterns as if they were physiology.

**Era and data quality.** All four files date from the 1980s, and diagnostic practice and patient populations have changed since. Data quality also differs by hospital. Switzerland and the VA are missing many measurements, which were filled with each hospital's median or most common value, and both hospitals recorded missing cholesterol values as 0 (all 123 Swiss patients and 49 VA patients, Section 3). Because cholesterol is 0 only at these two high-prevalence hospitals, the model can partly learn which hospital a patient came from through that column, which is a recording artifact and not a property of the patient.

### 9.2 How does the model perform across demographic groups?

**By sex (Section 8.4).** The model ranks men and women about equally well (ROC-AUC 0.91 for men and 0.89 for women) and misses only 1 of the 8 diseased women in the test set. Its errors for women fall the other way: 5 of the 12 women it flags are healthy, a precision of 58 percent against 79 percent for men. Only 8 diseased women are in the test set, so one patient moves female recall by 12.5 points and these figures should be read as warning signs.

**By age (Section 8.4b).** Younger diseased patients are missed most often, with recall of about 84 percent for patients aged 45 and under against about 97 percent for patients over 60. Older disease-free patients are over-referred most often, since 11 of the 13 disease-free patients over 60 are flagged. Part 1 predicted the first pattern, and the data now confirm it.

**By hospital (Section 8.5).** At the VA, 9 of the 10 disease-free test patients are flagged, so a hospital with a different patient mix would see very different false-alarm rates.

**What this means.** The overall recall of 93 percent hides groups that fare worse. Before deployment, sensitivity and specificity should be reported separately for each sex, age band, and hospital, on a sample large enough to measure them with confidence.

### 9.3 Is the model interpretable and fair? How transparent is its decision-making?

A Random Forest does not offer a single coefficient that tells a clinician why a patient scored high. Its feature importances (Section 8.6) show which inputs drive the model overall, led by asymptomatic chest pain, exercise-induced angina, ST depression, and peak heart rate, and they line up with the exploratory findings in Section 3.1. Sex accounts for about 5.5 percent of the importance. Global importances do not explain an individual score, so per-patient explanations, such as SHAP values, would be needed before clinicians could see the reasons behind each flag.

The model is proposed as decision support only. It ranks patients for angiography priority and never makes the referral decision itself. On fairness, the gaps by sex and age mean the model is not ready to deploy without safeguards, including clinician review of low scores for younger patients and women and routine monitoring of error rates by group.

**Data protection.** The UCI data were de-identified before public release and contain no names, dates, or record numbers. The notebook fetches the files from UCI at run time and stores no data in the GitHub repository. A deployed version would handle identifiable health information, so it would fall under HIPAA in the United States and the GDPR in Europe, which require a lawful basis for processing, minimum-necessary access, audit logging, and human review of automated decisions that significantly affect patients.

## 10. Real-World Application

**The use case.** The model is built for one realistic job: ranking patients who have completed a first-line workup so that a cardiology clinic can schedule angiography in order of risk. The score would be computed inside the electronic health record as soon as the stress test result is filed, using only fields the workup already captures, and shown to the cardiologist together with the inputs that drove it.

**How the score would be used.** The held-out test set shows how a three-tier schedule would work at the 0.42 threshold.

| Risk tier (predicted probability) | Test patients | With disease | Disease rate | Suggested next step |
|---|---|---|---|---|
| High (0.70 and above) | 79 | 73 | 92% | Earliest angiography slots |
| Intermediate (0.42 to 0.70) | 45 | 22 | 49% | Angiography scheduled in score order |
| Low (below 0.42) | 60 | 7 | 12% | Non-invasive test first, such as CT coronary angiography or a stress imaging study |

The high tier holds 43 percent of patients but 72 percent of all disease (73 of 102), so ordering the angiography list by score brings most diseased patients to the front of the queue. Directing low-risk patients to a non-invasive test first is consistent with the testing pathways in the current chest pain guideline (Gulati et al., 2021).

**Why the model is not a rule-out test.** About 12 percent of the low-risk tier (7 of 60 patients) still had significant disease. That rate is far too high to send those patients home without further testing, which is why the low tier leads to a cheaper test and not to discharge. The model changes the order and type of testing while leaving both the clinician and the confirmatory test in the pathway. Keeping a clinician who can review the basis for each score also places the tool within the category of clinical decision support software that the U.S. Food and Drug Administration (2022) describes.

**What deployment would require.** Section 8.5 showed that the model behaves very differently by hospital. At the VA it flagged 9 of 10 disease-free patients, so each adopting hospital would first need to test the model on several hundred of its own past patients, confirm sensitivity by sex and age band, and adjust the threshold or recalibrate the probabilities where needed (Van Calster et al., 2019). The data pipeline would need to record missing measurements as missing, not as 0, because Section 9 showed how placeholder values can leak hospital identity into the model. Once live, the clinic would monitor sensitivity, specificity, and the share of patients flagged each quarter, because referral patterns change over time. Reporting the model's development and validation according to the TRIPOD statement (Collins et al., 2015) would let a clinical governance committee judge it on the same terms as any other prediction tool.

**Who benefits.** Cardiologists get a consistent, explainable ordering of their referral list. Hospital quality leadership gets a measurable target, missed diagnoses by group, that can be tracked over time. Patients at high risk reach angiography sooner, and patients at low risk are more likely to start with a non-invasive test.

## 11. Team Collaboration Process

**Part 2 work**
Part 2 followed the same structure as Part 1, with each member owning part of the analysis. The work began from the three next steps agreed at the end of Part 1. One member expanded the dataset to all four hospitals and built the tuning and evaluation work in Sections 2 to 8, another added XGBoost as a fourth candidate model, and the remaining members integrated the sections, added the baseline and age-band checks, and wrote the problem framing, ethical analysis, real-world application, and conclusion. When the XGBoost results tied with the Random Forest, the team agreed to keep the Random Forest as the final model so that the evaluation in Section 8 remained complete and consistent.

**Key decision**
The biggest decision the team made together was to keep the Random Forest as the final model after XGBoost edged ahead of it. XGBoost was added late in the project and placed first with a cross-validated ROC-AUC of 0.8719, against 0.8716 for the Random Forest. Switching would have replaced the model behind every result in Sections 7.3 to 8, all of which had already been completed and interpreted. A lead of 0.0003 is smaller than the variation between cross-validation folds, so the two models perform the same in practice, and the team agreed the Random Forest's complete and consistent evaluation was worth more than the tiny difference. The decision is recorded openly in Section 7.1, where the code prints XGBoost as the top scorer and then states the team's final choice.

**Communication**
Our team used a collaborative and organized approach throughout the project so that both the technical analysis and this written report reflect decisions the whole group made together. At the beginning of the project we discussed the overall objective, selected the Heart Disease dataset, and agreed that the main goal would be to predict the presence of coronary artery disease using only the information available during the initial clinical workup. Responsibilities were then divided among team members according to the major stages of the analytics process, which meant that one member owned data exploration and cleaning, another owned preprocessing and the train-test split, another owned model development, evaluation, and tuning, and the remaining member owned the problem framing, the ethical analysis, and the assembly of this report, with each stage of the notebook building directly on the one before it.  


**Tools**   
Google Colab served as the primary coding environment because it allowed every team member to open the same Python notebook, test code, and compare results without maintaining separate local configurations, and because the notebook fetches the dataset directly from UCI at run time, everyone worked from identical data. GitHub was used for version control and to maintain an organized record of changes to the project files, with the notebook committed with outputs cleared so that differences between versions stay readable and a decisions log recording the choices that later sections depend on. Outlook email and Blackboard messages were the team’s main communication channels, and we used them to share project updates, discuss task responsibilities, exchange files and links, communicate deadlines, and follow up on decisions that required input from more than one person. Keeping those exchanges in writing meant that important information was documented and available to everyone, which mattered most when team members were working at different times.   


**Decisions**   
Team members communicated regularly to review findings, identify issues, and agree on methodological decisions before they were included in the final analysis. The clearest example is the choice of clinical variables, where the team discussed which predictors belonged in the model and agreed to exclude the fluoroscopy vessel count and the thallium scan because both are obtained late in the diagnostic process, after the point at which a referral model would be used. In Part 1 we also reviewed the importance of preventing data leakage by fitting all preprocessing inside the pipeline on training data only, selected evaluation metrics suited to a clinical setting, and agreed to emphasize recall because a false negative is a patient with disease whom the model would fail to refer. After evaluating the initial and tuned Random Forest models, the team compared the results and discussed the limitations that a small dataset, uneven demographic representation, and possible referral bias place on what the model can claim. Working through those decisions together in Colab, GitHub, and written communication allowed the team to coordinate responsibilities, keep everyone informed, and make each technical decision collaboratively from the start of the project to the end.



## 12. Final Thoughts and Conclusion

Part 2 set out to answer the three next steps from Part 1, and each produced a clear result. Comparing a majority-class baseline with four tuned models on ROC-AUC showed that the choice of algorithm matters little at this scale: Random Forest, XGBoost, Logistic Regression, and Gradient Boosting all finished between 0.865 and 0.872, far above the baseline's 0.50, and tuning lifted the Random Forest by about 0.015 over its default settings. Tuning the decision threshold proved to be the lever Part 1 was missing. At 0.42 the Random Forest caught 95 of 102 diseased patients in the 184-patient test set (93 percent recall), compared with 92 at the default 0.50, at a cost of 9 additional false alarms. Expanding the data to all four hospitals tripled the size of the test set and produced a test ROC-AUC of 0.92, with strong results at Cleveland and Hungary and much weaker specificity at the two high-prevalence hospitals.

The ethical analysis turned the Part 1 concerns into measurements. The model ranks men and women about equally well, but it is more likely to miss younger diseased patients and far more likely to flag healthy patients over 60. Together with the referral bias built into all four hospitals, these findings are why the model is proposed as a prioritization aid within a clinician-led referral pathway and not as a test that rules disease out.

The main limitations are that missing values were filled before the train-test split, that two hospitals recorded missing cholesterol as 0, that the female sample is small, and that all four datasets date from the 1980s. Every hospital also contributed training patients, so the results do not yet show how the model would perform at a new hospital. The most valuable next steps would be to fill missing values inside the training pipeline and treat placeholder zeros as missing, to train on three hospitals and test on the fourth as a true external check, to add per-patient explanations, and to validate the model on recent patients from a hospital's own records before any clinical use.

## References

Collins, G. S., Reitsma, J. B., Altman, D. G., & Moons, K. G. M. (2015). Transparent reporting of a multivariable prediction model for individual prognosis or diagnosis (TRIPOD): The TRIPOD statement. *Annals of Internal Medicine, 162*(1), 55–63. https://doi.org/10.7326/M14-0697

Detrano, R., Janosi, A., Steinbrunn, W., Pfisterer, M., Schmid, J.-J., Sandhu, S., Guppy, K. H., Lee, S., & Froelicher, V. (1989). International application of a new probability algorithm for the diagnosis of coronary artery disease. *The American Journal of Cardiology, 64*(5), 304–310. https://doi.org/10.1016/0002-9149(89)90524-9

Garcia, M., Mulvagh, S. L., Merz, C. N. B., Buring, J. E., & Manson, J. E. (2016). Cardiovascular disease in women: Clinical perspectives. *Circulation Research, 118*(8), 1273–1293. https://doi.org/10.1161/CIRCRESAHA.116.307547

Gulati, M., Levy, P. D., Mukherjee, D., Amsterdam, E., Bhatt, D. L., Birtcher, K. K., Blankstein, R., Boyd, J., Bullock-Palmer, R. P., Conejo, T., Diercks, D. B., Gentile, F., Greenwood, J. P., Hess, E. P., Hollenberg, S. M., Jaber, W. A., Jneid, H., Joglar, J. A., Morrow, D. A., . . . Shaw, L. J. (2021). 2021 AHA/ACC/ASE/CHEST/SAEM/SCCT/SCMR guideline for the evaluation and diagnosis of chest pain. *Circulation, 144*(22), e368–e454. https://doi.org/10.1161/CIR.0000000000001029

Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1988). *Heart disease* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X

U.S. Food and Drug Administration. (2022). *Clinical decision support software: Guidance for industry and Food and Drug Administration staff*. https://www.fda.gov/regulatory-information/search-fda-guidance-documents/clinical-decision-support-software

Van Calster, B., McLernon, D. J., van Smeden, M., Wynants, L., & Steyerberg, E. W. (2019). Calibration: The Achilles heel of predictive analytics. *BMC Medicine, 17*, Article 230. https://doi.org/10.1186/s12916-019-1466-7